In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



In [2]:
from modules.task_paths import load_and_register_tasks
tasks = load_and_register_tasks()
concrete_states = {}
sketches_by_idx = {}
for idx, task in enumerate(tasks):
    concrete_state = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
    for i, state in enumerate(concrete_state):
        sketches_by_idx[idx] = (task.desc, f"{task.workload_key}_{i}")
        concrete_states[f"{task.workload_key}_{i}"] = (task, state)

In [50]:
cs = SketchPolicy(tasks[400], params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()[0]
cs.transform_steps[-5].extent

2048

In [25]:
cs.transform_steps

[auto_scheduler.ComputeInlineStep(0x17ffef38), auto_scheduler.SplitStep(0x17f9d898), auto_scheduler.SplitStep(0x17f8cb48), auto_scheduler.SplitStep(0x17ff5768), auto_scheduler.SplitStep(0x17febe68), auto_scheduler.SplitStep(0x17ffeb28), auto_scheduler.SplitStep(0x17ffeb68), auto_scheduler.SplitStep(0x18015c88), auto_scheduler.ReorderStep(0x17fd9518), auto_scheduler.FollowSplitStep(0x17fd34c8), auto_scheduler.FollowSplitStep(0x180065a8), auto_scheduler.FollowSplitStep(0x17eb2d18), auto_scheduler.FollowSplitStep(0x17ffea08), auto_scheduler.ReorderStep(0x17ff9a88), auto_scheduler.ComputeAtStep(0x17fb7a28), auto_scheduler.CacheReadStep(0x17ff21c8), auto_scheduler.ComputeAtStep(0x17fdbc08), auto_scheduler.CacheReadStep(0x17414b38), auto_scheduler.ComputeAtStep(0x17fedc38), auto_scheduler.ComputeInlineStep(0x18001668), auto_scheduler.FuseStep(0x17abb908), auto_scheduler.AnnotationStep(0x1802c4d8), auto_scheduler.FuseStep(0x17fc4118), auto_scheduler.AnnotationStep(0x1802fd08), auto_scheduler.

In [ ]:
from modules.symbolic_state_bridge import build_symbolic_state
from modules.schedule_generator import ScheduleGenerator
sketch_idx = 178
print(sketches_by_idx[sketch_idx])
sample = list(concrete_states.values())[sketch_idx]
sym_state = build_symbolic_state(sample[0], sample[1])
sg = ScheduleGenerator(sym_state, task=sample[0], base_state=sample[1])
# sym_state

('vm_mod_fused_nn_conv2d_add_add_clip_divide_multiply_5', '["8b290ca0f393be6ffa536c91e9bafd22", [1, 15, 15, 80], [1, 1, 80, 184], [1, 1, 1, 184], [1, 15, 15, 184]]_0')


In [ ]:
initial_domains = {}
for name in sg._all_sp_names:
    step_idx = int(name.split("_")[1])
    extent = sg._sp_extents.get(step_idx)
    if extent is None:
        initial_domains[name] = 1
    else:
        initial_domains[name] = [1, extent]

initial_preview_sym_map = {name: None for name in sg.s.sym_map}
initial_fixed_from_domains = {}
for name, dom in initial_domains.items():
    if isinstance(dom, int):
        initial_preview_sym_map[name] = dom
        initial_fixed_from_domains[name] = dom
    elif isinstance(dom, list) and len(dom) == 2 and dom[0] == dom[1]:
        initial_preview_sym_map[name] = dom[0]
        initial_fixed_from_domains[name] = dom[0]

print("[Fixed From Domains Before Generation]")
print(initial_fixed_from_domains)

print("\n[Constraints Before Generation]")
print(
    sg.get_constraints_with_assignment_str(
        sym_map=initial_preview_sym_map,
        include_vars=True,
        include_eval=True,
    )
)


In [ ]:
import random

phase_entries = sg.get_var_order_phase_entries()

print("[Phase Order]")
for entry in phase_entries:
    shown_vars = entry["vars"] if entry["vars"] else ["<empty>"]
    print(entry["name"], shown_vars)

original_sym_map = dict(sg.s.sym_map)

for entry in phase_entries:
    print(f"\n=== {entry['name']} ===")
    prefix_state = sg.param_sampler.randomize_params_prefix(
        entry["name"],
        rng=random.Random(0),
        max_retries=1,
    )
    sg.s.sym_map.update(original_sym_map)

    preview_sym_map = {name: None for name in original_sym_map}
    preview_sym_map.update(prefix_state["fixed_values"])
    preview_sym_map.update(prefix_state["params"])

    print("[Phase Vars]")
    print(entry["vars"] if entry["vars"] else ["<empty>"])

    print("\n[Params At This Phase]")
    print(prefix_state["params"])

    print("\n[Fixed From Domains]")
    fixed_from_domains = {
        name: value
        for name, value in prefix_state["fixed_values"].items()
        if name not in prefix_state["params"]
    }
    print(fixed_from_domains)

    print("\n[Domains After Phase]")
    print(prefix_state["domains"])

    print("\n[Constraints After Phase]")
    print(
        sg.get_constraints_with_assignment_str(
            sym_map=preview_sym_map,
            include_vars=True,
            include_eval=True,
        )
    )
